# Chapter 8: The Eigenvalue Story

## Metric factor eigenvalues, condition number, and effective rank

Recall that each layer transition decomposes as $T^{(l)} = R^{(l)} P^{(l)}$, where $R$ is a rotor (rotation, grade-2) and $P$ is the metric factor (symmetric positive-definite, grade-0). The metric factor $P$ has real positive eigenvalues:

$$\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_k > 0$$

These eigenvalues are the **grade-0 content** of the layer transition. They determine which directions in representation space the model **amplifies** ($\sigma_i > 1$), **preserves** ($\sigma_i \approx 1$), or **suppresses** ($\sigma_i < 1$). While the rotor $R$ decides *where* information points, the metric factor $P$ decides *how much* of it survives.

Two key summary statistics capture the shape of this spectrum:

- **Condition number:** $\kappa = \sigma_1 / \sigma_k$ — the ratio of largest to smallest stretch. High $\kappa$ means the layer is highly selective, amplifying some directions far more than others.
- **Effective rank:** $\text{erank} = \exp\!\bigl(H(\hat{\sigma})\bigr)$ where $H(\hat{\sigma}) = -\sum_i \hat{\sigma}_i \log \hat{\sigma}_i$ is the entropy of the normalized eigenvalues $\hat{\sigma}_i = \sigma_i / \sum_j \sigma_j$. This measures how many directions carry significant weight — a "soft" count of active dimensions.

**Important caveat: well-determined directions only.** Because the layer operator is estimated from $T$ tokens in a $p$-dimensional space, only $r \leq 2T$ singular values are well-determined. The rest sit at machine-epsilon and reflect the underdetermined null space, not the layer's geometry. Including them inflates $\kappa$ to absurd values ($10^9$–$10^{11}$) and deflates the effective rank. Throughout this notebook we filter to singular values above $0.1\%$ of the maximum.

**By the end of this chapter you will be able to:**
- Read and interpret eigenvalue spectra of the metric factor
- Compute and interpret condition number and effective rank using well-determined directions
- Understand the different roles of grade-0 (stretching) and grade-2 (rotation)

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga
from layer_time_ga.decomposition import extract_rotor_field, extract_metric_field

print("Imports OK")
print(f"NumPy {np.__version__}")

In [ ]:
# Load model and run GA analysis
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")

prompt = "The capital of France is"
result = ltg_ga.analyse(prompt, model=model, compute_dependency=False)

# Also extract the full metric field for eigenvalue access
metric_field = extract_metric_field(result.H_whitened, skip_first=False)

print(f"Model: {model.name}")
print(f"Prompt: \"{prompt}\"")
print(f"Layers: {result.n_layers}, Whitened dim: {result.k}")
print(f"Metric field: {len(metric_field)} layer transitions")
print(f"Rotor field: {len(result.rotor_field.decompositions)} decompositions")

## Reading the Eigenvalue Spectrum

The eigenvalue spectrum of $P^{(l)}$ is a fingerprint of what the layer does to representation geometry:

| Eigenvalue $\sigma_i$ | Meaning |
|:---|:---|
| $\sigma_i = 1$ | **Identity** — this direction passes through unchanged |
| $\sigma_i \gg 1$ | **Amplified** — the model is boosting signal in this direction |
| $\sigma_i \ll 1$ | **Suppressed** — the model is discarding information along this direction |

A spectrum clustered tightly around 1 means the layer is "gentle" — mostly preserving the representation with small tweaks. A spectrum with eigenvalues spanning a wide range means the layer is aggressively reshaping the representation.

Let us plot the eigenvalue spectra for three representative layers — early, middle, and late — using only well-determined singular values, to see how metric selectivity changes with depth.

In [ ]:
# --- Eigenvalue spectra for early, middle, late layers ---
# Filter to well-determined singular values only

SV_THRESH = 1e-3  # keep SVs above 0.1% of the max

n_trans = len(metric_field)
early_idx = 0
mid_idx = n_trans // 2
late_idx = n_trans - 1

layer_indices = [early_idx, mid_idx, late_idx]
layer_labels = ['Early', 'Middle', 'Late']
colors = ['#4477AA', '#228833', '#EE6677']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, l_idx, label, color in zip(axes, layer_indices, layer_labels, colors):
    sv = np.sort(metric_field[l_idx]['singular_values'])[::-1]
    sv_good = sv[sv > SV_THRESH * sv[0]]
    kappa = sv_good[0] / sv_good[-1]
    p = sv_good / sv_good.sum()
    erank = np.exp(-np.sum(p * np.log(p + 1e-30)))

    ax.bar(range(len(sv_good)), sv_good, width=1.0, color=color)
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.7, linewidth=1, label='$\\sigma = 1$ (identity)')
    ax.set_xlabel('Rank index $i$')
    ax.set_title(f'{label} (layer {l_idx}, $\\kappa$={kappa:.1f}, r={len(sv_good)})')
    ax.set_ylabel('Singular value $\\sigma_i$')
    ax.legend(fontsize=8)

fig.suptitle('Singular-Value Spectra of the Metric Factor (well-determined only)', fontsize=13, y=1.02)
fig.tight_layout()

save_path = 'ch08_eigenvalue_spectra.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

print("\nRed dashed line = identity (no stretch). Bars above are amplified; below are suppressed.")
print("Note: only well-determined singular values shown (above 0.1% of max).")

## Condition Number and Effective Rank Across Layers

Now let us track both summary statistics across all layers, using only well-determined singular values:

- **Condition number $\kappa$** tells us how *selective* the layer is — how much it differentiates between its most-amplified and most-suppressed directions. A layer with $\kappa \approx 1$ treats all directions equally (isotropic). A layer with $\kappa \gg 1$ is focusing on a narrow subspace.

- **Effective rank** tells us how *many* directions carry significant weight in the eigenvalue distribution. Low effective rank means the layer concentrates its stretching on few directions; high effective rank means the stretching is distributed broadly.

In Qwen2.5-7B, the pattern is the opposite of what you might expect: **early layers are the most selective** ($\kappa \approx 5.6$ at layer 0), while late layers converge toward isometry ($\kappa \approx 1.0$). The early layers establish the representation's geometry — aggressively amplifying and suppressing directions — while late layers preserve the structure already in place.

In [ ]:
# --- Condition number and effective rank across layers (well-determined only) ---

kappas_eff = []
eranks_eff = []
for mf in metric_field:
    sv = np.sort(mf['singular_values'])[::-1]
    sv_good = sv[sv > SV_THRESH * sv[0]]
    kappas_eff.append(sv_good[0] / sv_good[-1])
    p = sv_good / sv_good.sum()
    eranks_eff.append(np.exp(-np.sum(p * np.log(p + 1e-30))))
kappas_eff = np.array(kappas_eff)
eranks_eff = np.array(eranks_eff)
layers = np.arange(len(kappas_eff))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Condition number
ax1.plot(layers, kappas_eff, color='#AA3377', linewidth=2, marker='o', markersize=3)
ax1.fill_between(layers, kappas_eff, alpha=0.15, color='#AA3377')
ax1.set_xlabel('Layer transition')
ax1.set_ylabel('Condition number $\\kappa = \\sigma_1 / \\sigma_k$')
ax1.set_title('Metric Selectivity (well-determined directions)')
peak_kappa = np.argmax(kappas_eff)
ax1.axvline(peak_kappa, color='grey', linestyle='--', alpha=0.5,
            label=f'Peak at layer {peak_kappa} ($\\kappa$ = {kappas_eff[peak_kappa]:.1f})')
ax1.legend(fontsize=9)

# Panel 2: Effective rank
ax2.plot(layers, eranks_eff, color='#66CCEE', linewidth=2, marker='o', markersize=3)
ax2.fill_between(layers, eranks_eff, alpha=0.15, color='#66CCEE')
ax2.set_xlabel('Layer transition')
ax2.set_ylabel('Effective rank')
ax2.set_title('Metric Dimensionality (well-determined directions)')
min_erank = np.argmin(eranks_eff)
ax2.axvline(min_erank, color='grey', linestyle='--', alpha=0.5,
            label=f'Minimum at layer {min_erank} (erank = {eranks_eff[min_erank]:.1f})')
ax2.legend(fontsize=9)

fig.suptitle('Grade-0 Summary: Condition Number and Effective Rank', fontsize=13, y=1.02)
fig.tight_layout()

save_path = 'ch08_kappa_erank.png'
fig.savefig(save_path, bbox_inches='tight', dpi=150)
print(f"Saved: {save_path}")
plt.show()

print(f"\nCondition number range: [{kappas_eff.min():.1f}, {kappas_eff.max():.1f}]")
print(f"Effective rank range: [{eranks_eff.min():.1f}, {eranks_eff.max():.1f}]")
print(f"Peak selectivity at layer {peak_kappa} — early layers stretch most.")

## Why Grade-0 Matters

Rotation and stretching play fundamentally different roles, and this follows from a simple mathematical fact:

**Rotation preserves norms. Stretching changes norms.**

If $R$ is an orthogonal matrix (a rotor in our GA framework), then for any vector $v$:
$$\|Rv\| = \|v\|$$

The rotor shuffles directions but cannot amplify or suppress any component. It is *isometric* — it preserves the geometry of the representation.

The metric factor $P$, on the other hand, acts by stretching:
$$\|Pv\| \neq \|v\| \quad \text{(in general)}$$

The eigenvalues of $P$ directly control how much each component of $v$ is amplified or suppressed. During **backpropagation**, gradient magnitudes are controlled by the singular values of the Jacobian — and those singular values come from $P$, not from $R$.

This means:
- Modifying the **eigenvalues of $P$** (grade-0) changes the magnitude of gradient flow, directly affecting which features survive.
- Modifying the **rotation planes of $R$** (grade-2) redirects gradient flow without changing its magnitude.

Both grades are essential: rotation *steers*, stretching *selects*. Chapter 12 (Which Planes Matter?) will quantify which role has more impact on the model's output using dependency density.

Let us verify the norm-preservation property with a direct computation.

In [ ]:
# --- Demonstration: rotation preserves norms, stretching changes norms ---

# Pick the most selective layer (highest well-determined kappa)
demo_layer = int(np.argmax(kappas_eff))
decomp = result.rotor_field.decompositions[demo_layer]

R_matrix = decomp.rotor.matrix   # Orthogonal (rotation)
P_matrix = decomp.metric         # Symmetric PD (stretch)

# Create a random unit vector in the whitened space
rng = np.random.default_rng(42)
v = rng.standard_normal(R_matrix.shape[0])
v = v / np.linalg.norm(v)

# Apply rotor (rotation)
v_rotated = R_matrix @ v

# Apply metric factor (stretching)
v_stretched = P_matrix @ v

# Apply the full transition T = R @ P
v_full = R_matrix @ (P_matrix @ v)

print(f"=== Norm Preservation Test (Layer {demo_layer}, kappa = {kappas_eff[demo_layer]:.1f}) ===")
print()
print(f"  Original vector ||v||          = {np.linalg.norm(v):.6f}")
print(f"  After rotation  ||Rv||         = {np.linalg.norm(v_rotated):.6f}")
print(f"  After stretching ||Pv||        = {np.linalg.norm(v_stretched):.6f}")
print(f"  After full transition ||RPv||  = {np.linalg.norm(v_full):.6f}")
print()
print(f"  Norm change from rotation:  {abs(np.linalg.norm(v_rotated) - np.linalg.norm(v)):.2e}  (essentially zero)")
print(f"  Norm change from stretching: {abs(np.linalg.norm(v_stretched) - np.linalg.norm(v)):.4f}  (significant!)")
print()
print("Rotation preserves the norm exactly — it is an isometry.")
print("Stretching changes the norm — it amplifies or suppresses components.")

## The Selectivity Profile

The data reveals a clear pattern: **early layers are most selective**, not late layers.

- Layer 0 has $\kappa \approx 5.6$: its top singular value is $\sim 4.0$, amplifying some directions by $4\times$ while compressing others to $\sim 0.7$. This is where the representation's geometry is established.
- By mid-network, the spectrum narrows ($\kappa \approx 1.4$).
- By the final layer, all well-determined singular values sit at $\approx 1.0$ ($\kappa \approx 1.0$): a near-isometry that preserves the structure already in place.

This raises an important question: if grade-0 is largest in early layers but the model's prediction is made at the final layer, does grade-0 still matter for the output? The answer requires **dependency density** — a gradient-based measure of how sensitive the output is to each layer's geometry. We defer this analysis to Chapter 12, where the interaction between the eigenvalue spectrum and dependency will resolve the question quantitatively.

Let us examine the most selective layer in detail.

In [ ]:
# --- Most selective layer: well-determined singular values ---

most_selective = int(np.argmax(kappas_eff))
sv = np.sort(metric_field[most_selective]['singular_values'])[::-1]
sv_good = sv[sv > SV_THRESH * sv[0]]

print(f"=== Most Selective Layer: Transition {most_selective} ===")
print(f"Condition number (well-determined): {kappas_eff[most_selective]:.2f}")
print(f"Effective rank (well-determined): {eranks_eff[most_selective]:.1f}")
print(f"Well-determined directions: {len(sv_good)} out of {len(sv)} total")
print()

print("Top-5 singular values (most amplified directions):")
for i in range(min(5, len(sv_good))):
    print(f"  sigma_{i+1:3d} = {sv_good[i]:.4f}  ({sv_good[i]:.1f}x amplification)")

print()
print("Bottom-5 singular values (most compressed directions):")
for i in range(max(0, len(sv_good) - 5), len(sv_good)):
    print(f"  sigma_{i+1:3d} = {sv_good[i]:.4f}  ({sv_good[i]:.2f}x)")

print()
print(f"Ratio sigma_1 / sigma_r = {sv_good[0] / sv_good[-1]:.1f}")
print(f"The early layers establish representation geometry by selectively")
print(f"amplifying and compressing directions. Late layers preserve it.")

## Exercises

1. **Eigenvalue spectrum shape.** For each layer, compute the fraction of well-determined eigenvalues that are above 1.0 (amplified directions) vs. below 1.0 (suppressed directions). Plot this fraction across layers. Does the balance shift from early to late layers?

2. **Grade-0 energy vs. grade-2 energy.** For each layer transition, compute $\|P - I\|_F$ (how far the metric factor is from the identity) and $\|B\|_F$ (the bivector norm) using only well-determined directions. Plot both on the same chart. At which layers does each dominate?

3. **Norm amplification experiment.** Take 1000 random unit vectors. For the most selective layer, compute $\|Pv\|$ for each. Plot the distribution. How does it relate to the eigenvalue spectrum? Now do the same for $\|Rv\|$ — what do you expect?

4. **Effective rank as a complexity measure.** Compare the well-determined effective rank profile across two prompts of different complexity (e.g., "Hello" vs. a long technical sentence). Do more complex prompts produce different effective rank profiles?

5. **Where does selectivity matter most?** The most selective layer (highest $\kappa$) is early in the network. But does it matter most for the output? Chapter 12 introduces dependency density $D_l$ to answer this. As a preview: compute $\kappa^{(l)} \times \|P^{(l)} - I\|_F$ across layers. Where is the product largest?